# Betting performance

EPL and Champions League P/L; EPL post-match xG diagnostics.

Copy this template to a `.local.ipynb` file before running it. Run `bet-analytics` from the project directory to refresh the cache first; this notebook does not download data. An `INCOMPLETE SYNC` message means the refresh failed in part.

Expand the preparation cell to choose `SEASON` or pass `user=` to `load_notebook_data` when using multiple profiles. Results and exported charts contain personal betting history. Post-match xG benchmarks are retrospective diagnostics, not evidence of a profitable strategy.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os
import sys

now_utc = datetime.now(timezone.utc)
SEASON = now_utc.year if now_utc.month >= 7 else now_utc.year - 1
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, *cwd.parents) if (candidate / 'pyproject.toml').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Start JupyterLab from the football betting project directory.')

src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
matplotlib_config = PROJECT_ROOT / 'data' / 'matplotlib-cache'
matplotlib_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(matplotlib_config))

try:
    import httpx  # imported here to give a useful kernel error before project imports
except ModuleNotFoundError as error:
    raise RuntimeError(
        f'Kernel {sys.executable} is missing the project dependencies. '
        'Run: python -m pip install -e ".[notebook]" from this repository.'
    ) from error

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from polymarket_bot.notebook_data import load_notebook_data

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

DATABASE = PROJECT_ROOT / 'data' / 'bet_history.sqlite3'
data = load_notebook_data(DATABASE, season=SEASON)
bets = data['bets'].copy()
trades = data['trades'].copy()
closed_positions = data['closed_positions'].copy()
xg_matches = data['xg_matches'].copy()
address = data['profile_address']
print(f"Loaded {len(trades)} cached trades from {DATABASE}")
print(f"Latest stored trade: {trades['placed_at'].max() if not trades.empty else 'none'}")

season_start = pd.Timestamp(year=SEASON, month=7, day=1, tz='UTC')
season_end = pd.Timestamp(year=SEASON + 1, month=7, day=1, tz='UTC')

outside_window = int((~trades['placed_at'].between(season_start, season_end, inclusive='left')).sum())
non_supported = int((~trades['event_slug'].str.casefold().str.startswith(('epl-', 'ucl-', 'champions-league-', 'uefa-champions-league-'))).sum())
duplicate_trades = int(trades['id'].duplicated().sum())
invalid_economics = int((
    ~np.isfinite(trades[['price', 'size', 'usdc_size']]).all(axis=1)
    |     ~trades['price'].between(0, 1, inclusive='both')
    | (trades['size'] <= 0)
    | (trades['usdc_size'] < 0)
).sum())
mixed_results = int((
    bets.groupby('selection_key', dropna=False)['result'].nunique(dropna=False) > 1
).sum())
unattributed_settlements = int((
    bets['hold_pnl_usd'].notna() & bets['resolution_source'].isna()
).sum())
wrong_xg_season = int((xg_matches['season'] != SEASON).sum())
buy_fixture_match_ids = set(bets['match_id'].dropna())
unrepresented_xg = int((~xg_matches['match_id'].isin(buy_fixture_match_ids)).sum())

scope_checks = pd.DataFrame([
    ('Trade dates', outside_window, f'Expected [{season_start.date()}, {season_end.date()})'),
    ('Supported event slugs', non_supported, 'Every imported event_slug must use an EPL or Champions League prefix'),
    ('Unique trade IDs', duplicate_trades, 'No duplicate API fills'),
    ('Price / size fields', invalid_economics, '0 ≤ price ≤ 1, shares > 0, cost ≥ 0'),
    ('Selection settlement', mixed_results, 'Repeated fills must agree on result'),
    ('Resolution provenance', unattributed_settlements, 'Every settled fill must identify its resolution source'),
    ('xG season', wrong_xg_season, f'Every xG row must belong to {SEASON}/{str(SEASON + 1)[-2:]}'),
    ('xG fixture scope', unrepresented_xg, 'Every xG row must match a represented BUY fixture'),
], columns=['check', 'issues', 'expectation'])
scope_checks.insert(1, 'status', np.where(scope_checks['issues'].eq(0), 'PASS', 'REVIEW'))

if (scope_checks['issues'] > 0).any():
    raise AssertionError('Scope or integrity checks failed; review the rows above before interpreting results.')

def single_value(values):
    observed = pd.Series(values).dropna().unique()
    if len(observed) == 0:
        return pd.NA
    return observed[0] if len(observed) == 1 else 'MIXED'

selections = (
    bets.groupby('selection_key', as_index=False, dropna=False)
    .agg(
        asset=('asset', 'first'),
        event_slug=('event_slug', single_value),
        market_title=('market_title', single_value),
        market_type=('market_type', single_value),
        market_role=('market_role', single_value),
        selection=('selection', single_value),
        direction_team=('direction_team', single_value),
        direction_opponent=('direction_opponent', single_value),
        placed_at=('placed_at', 'min'),
        kickoff=('kickoff', 'first'),
        fixture=('fixture', single_value),
        home_team=('home_team', single_value),
        away_team=('away_team', single_value),
        fills=('trade_id', 'count'),
        shares=('shares', 'sum'),
        amount_paid=('cost_usd', 'sum'),
        result=('result', single_value),
        resolution_source=('resolution_source', single_value),
        hold_pnl=('hold_pnl_usd', lambda values: values.sum(min_count=1)),
        home_goals=('home_goals', 'first'),
        away_goals=('away_goals', 'first'),
        post_home_xg=('post_home_xg', 'first'),
        post_away_xg=('post_away_xg', 'first'),
        direction_goals=('direction_goals', 'first'),
        opponent_goals=('opponent_goals', 'first'),
        direction_xg=('direction_xg', 'first'),
        opponent_xg=('opponent_xg', 'first'),
        direction_xg_diff=('direction_xg_diff', 'first'),
    )
)
selections['weighted_entry'] = selections['amount_paid'].div(selections['shares'])
selections['settled_amount_paid'] = selections['amount_paid'].where(selections['hold_pnl'].notna())

def describe_position(row):
    role = None if pd.isna(row['market_role']) else row['market_role']
    selection = None if pd.isna(row['selection']) else row['selection']
    proposition = (
        row['home_team'] if role == 'home'
        else row['away_team'] if role == 'away'
        else 'Draw' if role == 'draw'
        else row['market_title']
    )
    if selection == 'YES':
        return str(proposition)
    if selection == 'NO' and role in {'home', 'away'}:
        return f'{proposition} not to win'
    if selection == 'NO' and role == 'draw':
        return 'No draw'
    return f'{proposition} — {selection or "unknown outcome"}'

selections['position'] = pd.Series(
    [describe_position(row) for _, row in selections.iterrows()], index=selections.index, dtype='object'
)
selections = selections.sort_values(['placed_at', 'selection_key']).reset_index(drop=True)

fixture_keys = ['home_team', 'away_team', 'kickoff']
represented_fixtures = len(bets.dropna(subset=fixture_keys)[fixture_keys].drop_duplicates())
overview = pd.DataFrame([
    ('All EPL/UCL trade fills', len(trades)),
    ('BUY fills analyzed', len(bets)),
    ('SELL fills retained for audit', int(trades['side'].eq('SELL').sum())),
    ('Economic selections', len(selections)),
    ('Distinct EPL/UCL events', int(trades['event_slug'].nunique())),
    ('Represented match fixtures', represented_fixtures),
    ('Completed represented fixtures with xG', len(xg_matches)),
    ('Closed-position records', len(closed_positions)),
], columns=['measure', 'count'])

settled = selections.dropna(subset=['hold_pnl']).copy()
gross_buy_cost = selections['amount_paid'].sum()
settled_cost = settled['settled_amount_paid'].sum()
hold_pnl = settled['hold_pnl'].sum()
hold_roi = hold_pnl / settled_cost if settled_cost else np.nan
realized_pnl = closed_positions['realized_pnl'].sum() if not closed_positions.empty else 0.0

kpis = pd.DataFrame([
    ('Economic selections', f'{len(selections):,}', 'Unique tokens/positions after fill aggregation'),
    ('Gross BUY cost', f'${gross_buy_cost:,.2f}', 'All selected-season BUY fills'),
    ('Settled selections', f'{len(settled):,}', 'Official or score-fallback settlements; inspect resolution_source'),
    ('Settled BUY cost', f'${settled_cost:,.2f}', 'Denominator for hypothetical hold ROI'),
    ('Hypothetical hold P/L', f'${hold_pnl:+,.2f}', 'BUY shares held through settlement'),
    ('Hypothetical hold ROI', f'{hold_roi:+.1%}' if pd.notna(hold_roi) else 'n/a', 'Hold P/L ÷ settled BUY cost'),
    ('Closed-position realized P/L', f'${realized_pnl:+,.2f}', 'Closed-position subset only; see reconstructed P/L above for settled holdings'),
], columns=['metric', 'value', 'definition'])

market_summary = (
    selections.groupby('market_type', dropna=False)
    .agg(
        selections=('selection_key', 'count'),
        fills=('fills', 'sum'),
        shares=('shares', 'sum'),
        gross_buy_cost=('amount_paid', 'sum'),
        settled_selections=('hold_pnl', lambda values: int(values.notna().sum())),
        settled_cost=('settled_amount_paid', lambda values: values.sum(min_count=1)),
        hold_pnl=('hold_pnl', lambda values: values.sum(min_count=1)),
    )
    .sort_values('gross_buy_cost', ascending=False)
)
market_summary['weighted_entry'] = market_summary['gross_buy_cost'].div(market_summary['shares'])
market_summary['hold_roi'] = market_summary['hold_pnl'].div(market_summary['settled_cost'])

fixture_xg = xg_matches.sort_values('kickoff_utc').copy()
fixture_xg['fixture'] = fixture_xg['home_team'] + ' vs ' + fixture_xg['away_team']
fixture_xg['score'] = (
    fixture_xg['home_goals'].astype('Int64').astype(str)
    + '–' + fixture_xg['away_goals'].astype('Int64').astype(str)
)
fixture_xg['xg_score'] = pd.Series(
    [f'{row.home_xg:.2f}–{row.away_xg:.2f}' for row in fixture_xg.itertuples()],
    index=fixture_xg.index, dtype='object',
)
fixture_xg['total_xg'] = fixture_xg['home_xg'] + fixture_xg['away_xg']
fixture_xg['home_xg_diff'] = fixture_xg['home_xg'] - fixture_xg['away_xg']

directional_xg = selections.dropna(
    subset=['direction_team', 'direction_opponent', 'direction_xg_diff']
).copy()
directional_xg['direction_score'] = (
    directional_xg['direction_goals'].astype('Int64').astype(str)
    + '–' + directional_xg['opponent_goals'].astype('Int64').astype(str)
)
directional_xg = directional_xg.sort_values(['kickoff', 'placed_at'])

In [ ]:
from polymarket_bot.performance import plot_performance_dashboard

performance = plot_performance_dashboard(data, output_directory=PROJECT_ROOT / 'data' / 'performance_plots')
directional_matches = performance['directions']
pnl_events = performance['pnl']['events']
pnl_daily = performance['pnl']['daily']
open_holdings = performance['pnl']['open_holdings']
performance_warnings = performance['pnl']['warnings']
xg_benchmark_selections = performance['xg_benchmark']['selections']
xg_benchmark_daily = performance['xg_benchmark']['daily']
xg_benchmark_fill_audit = performance['xg_benchmark']['fill_audit']
xg_benchmark_exclusions = performance['xg_benchmark']['exclusions']

# DataFrames remain available for manual inspection, including bets, trades,
# selections, xg_matches, scope_checks, kpis, market_summary, directional_matches,
# pnl_events, pnl_daily, open_holdings, and the xg_benchmark_* frames.
for figure in performance['figures'].values():
    plt.close(figure)


In [ ]:
display(performance['figures']['01_realized_pnl'])

In [ ]:
display(performance['figures']['02_outcomes_vs_xg'])

In [ ]:
display(performance['figures']['03_cumulative_xg'])

In [ ]:
display(performance['figures']['04_per_match_goals_xg'])

In [ ]:
display(performance['figures']['05_xg_benchmark_roi'])

In [ ]:
display(performance['figures']['06_xg_benchmark_pnl'])